# CS453 — Lab 06: Graph Search with BFS and DFS
## Week 4 · Python 3 · Google Colab or Jupyter

Misurata University · Faculty of Information Technology · Computer Science Department  
Ali Elzaeilik · Autumn 2026

Lab 04 completed Python preparation and Lab 05 represented problems with states, valid actions, transitions and goal tests. This lab implements uninformed search on a small directed graph using familiar Python collections and iterative functions.

A **visiting order** records which states are processed. A **path** records connected states leading from the start to the goal. A Boolean result reports only whether the goal was found. These outputs answer different questions.


## 1. Graph representation and successors

Each dictionary key names a state. Its value is the ordered list of states reachable in one step. An empty list means there are no outgoing edges. The graph is directed: a connection in one direction does not imply a connection back.

The two routes from S to G are `S → B → G` (2 actions) and `S → A → C → F → G` (4 actions). Every edge has cost 1. Neighbour order is significant when deciding which branch is processed first.


In [1]:
graph = {
    "S": ["A", "B"],
    "A": ["C"],
    "B": ["G"],
    "C": ["F"],
    "F": ["G"],
    "G": []
}

print(graph["S"])
print(graph["A"])
print(graph["G"])
print(graph["B"])

for neighbour in graph["S"]:
    print(neighbour)


['A', 'B']
['C']
[]
['G']
A
B


## 2. Recording visited states and testing a cycle

A set stores unique values and supports membership tests. A visited record prevents repeated processing when a state is reachable by more than one route or a cycle leads back to an earlier state.

The cycle example uses a separate graph so the main graph and its expected orders remain unchanged. The lists are copied too, rather than sharing mutable lists between the two dictionaries. Adding F→S creates the cycle `S → A → C → F → S`.

Without repeat detection, exploration may continue indefinitely if no reachable goal stops it. A reachable goal can still stop some searches, so a cycle alone does not imply that every goal-seeking run loops forever.


In [2]:
visited = set()
visited.add("S")
visited.add("S")
print("unique recorded states:", len(visited))
print("S already visited?", "S" in visited)

cycle_graph = {}
for state, neighbours in graph.items():
    cycle_graph[state] = list(neighbours)
cycle_graph["F"] = ["G", "S"]
print("original F neighbours:", graph["F"])
print("cycle F neighbours:", cycle_graph["F"])


unique recorded states: 1
S already visited? True
original F neighbours: ['G']
cycle F neighbours: ['G', 'S']


## 3. A queue for BFS

`collections.deque` is a standard Python collection. `append` adds at the back and `popleft` removes the front, creating first-in, first-out (FIFO) order. The queue is called the frontier because it contains states waiting to be processed.

`while frontier` repeats while work remains. Empty collections have a false truth value, so this condition prevents removing an item from an empty queue. `list(frontier)` converts the deque to a list for display.


In [3]:
from collections import deque
frontier = deque(["S"])
current = frontier.popleft()
frontier.append("A")
frontier.append("B")
print(current, list(frontier))


S ['A', 'B']


## 4. BFS reachability

BFS removes the oldest pending state. It processes shallow states before deeper ones. This function prints its visiting order and returns `True` when it processes the goal, or `False` after the queue is exhausted.

This introductory version records a state when it is **processed**. `continue` skips duplicate entries without processing them again. Duplicates can temporarily wait in the queue; the path-returning BFS later records states on **discovery** instead to prevent duplicate enqueueing.

`graph.get(current, [])` treats a missing dictionary entry as having no outgoing connections. `return False` belongs outside the loop: failure is known only when all waiting work has been exhausted. Each call creates its own visited set.


In [4]:
from collections import deque

def bfs(graph, start, goal):
    frontier = deque([start])
    visited = set()
    while frontier:
        current = frontier.popleft()
        if current in visited:
            continue
        visited.add(current)
        print("Visiting:", current)
        if current == goal:
            return True
        for neighbour in graph.get(current, []):
            if neighbour not in visited:
                frontier.append(neighbour)
    return False


In [5]:
print("Goal G:", bfs(graph, "S", "G"))
print("Goal F:", bfs(graph, "S", "F"))
print("Goal X:", bfs(graph, "S", "X"))
print("Start equals goal:", bfs(graph, "S", "S"))
print("Cycle, missing goal:", bfs(cycle_graph, "S", "X"))


Visiting: S
Visiting: A
Visiting: B
Visiting: C
Visiting: G
Goal G: True
Visiting: S
Visiting: A
Visiting: B
Visiting: C
Visiting: G
Visiting: F
Goal F: True
Visiting: S
Visiting: A
Visiting: B
Visiting: C
Visiting: G
Visiting: F
Goal X: False
Visiting: S
Start equals goal: True
Visiting: S
Visiting: A
Visiting: B
Visiting: C
Visiting: G
Visiting: F
Cycle, missing goal: False


For G, BFS visits `S, A, B, C, G` and returns `True`. F is waiting but has not been processed when G is found. For F, the order is `S, A, B, C, G, F`. For the absent goal X, that same full order ends in `False`.

A missing goal is different from a missing dictionary key. The goal is merely a value used in a comparison; it need not have an entry for an unsuccessful search to finish cleanly.


## 5. A stack for iterative DFS

A list used with `append` and `pop()` is a last-in, first-out (LIFO) stack. The rightmost element is the top. DFS follows the most recently scheduled branch before returning to earlier alternatives.

For neighbours `[A, B]`, pushing A then B makes B leave first. Iterating over `reversed(...)` pushes B then A, allowing A to leave first. `reversed` does not modify the stored neighbour list.

This DFS uses a loop, not recursion. Like the introductory BFS, it marks states when they are processed and skips repeated pending entries.


In [6]:
stack = ["S"]
current = stack.pop()
for neighbour in reversed(graph[current]):
    stack.append(neighbour)
print("stack, bottom to top:", stack)
print("next state:", stack.pop())


stack, bottom to top: ['B', 'A']
next state: A


In [7]:
def dfs(graph, start, goal):
    stack = [start]
    visited = set()
    while stack:
        current = stack.pop()
        if current in visited:
            continue
        visited.add(current)
        print("Visiting:", current)
        if current == goal:
            return True
        for neighbour in reversed(graph.get(current, [])):
            if neighbour not in visited:
                stack.append(neighbour)
    return False


In [8]:
print("DFS result:", dfs(graph, "S", "G"))
print("Unreachable goal:", dfs(graph, "S", "X"))
print("Cycle, missing goal:", dfs(cycle_graph, "S", "X"))


Visiting: S
Visiting: A
Visiting: C
Visiting: F
Visiting: G
DFS result: True
Visiting: S
Visiting: A
Visiting: C
Visiting: F
Visiting: G
Visiting: B
Unreachable goal: False
Visiting: S
Visiting: A
Visiting: C
Visiting: F
Visiting: G
Visiting: B
Cycle, missing goal: False


Left-first DFS visits `S, A, C, F, G`. B remains waiting when the goal is reached. Removing `reversed` changes the first branch and produces `S, B, G` on this graph.

Both baseline BFS and left-first DFS process five states before reaching G, but their solution paths have different lengths. Visiting count is not path cost. The Boolean functions themselves do not return paths.


In [9]:
def dfs_right_first(graph, start, goal):
    stack = [start]
    visited = set()
    while stack:
        current = stack.pop()
        if current in visited:
            continue
        visited.add(current)
        print("Visiting:", current)
        if current == goal:
            return True
        for neighbour in graph.get(current, []):
            if neighbour not in visited:
                stack.append(neighbour)
    return False

print("Without reversed:", dfs_right_first(graph, "S", "G"))


Visiting: S
Visiting: B
Visiting: G
Without reversed: True


## 6. Parent maps and path reconstruction

A parent dictionary records which earlier state led to a state. The start has parent `None`, a sentinel marking the root. For BFS's short route, G has parent B, and B has parent S.

Following these parents produces `[G, B, S]`. Reversing the list produces `[S, B, G]`, a route from the start to the goal. `path.reverse()` changes the list in place and returns `None`; assigning its return value to `path` would lose the list.

The helper is called only after search reaches the goal and constructs a valid parent map.


In [10]:
def reconstruct_path(parent, goal):
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = parent[current]
    path.reverse()
    return path


In [11]:
example_parent = {"S": None, "B": "S", "G": "B"}
print(reconstruct_path(example_parent, "G"))


['S', 'B', 'G']


## 7. BFS returning a path

`bfs_path` marks states when they are **discovered and enqueued**. This ensures that each state enters the queue once and that its first parent is not overwritten by a later route. FIFO order makes that first route a route with the fewest edges.

The visited set contains the start immediately. New neighbours are recorded, given a parent, and appended to the queue. Reaching the goal calls `reconstruct_path`; exhausting the queue returns `None`.

For uniform positive edge costs, a path with the fewest edges also has the lowest cost. If edges have different costs, this BFS does not guarantee the cheapest route.


In [12]:
def bfs_path(graph, start, goal):
    frontier = deque([start])
    visited = {start}
    parent = {start: None}
    while frontier:
        current = frontier.popleft()
        if current == goal:
            return reconstruct_path(parent, current)
        for neighbour in graph.get(current, []):
            if neighbour not in visited:
                visited.add(neighbour)
                parent[neighbour] = current
                frontier.append(neighbour)
    return None


In [13]:
print("BFS path:", bfs_path(graph, "S", "G"))
print("No path:", bfs_path(graph, "S", "X"))
print("Already at goal:", bfs_path(graph, "S", "S"))


BFS path: ['S', 'B', 'G']
No path: None
Already at goal: ['S']


## 8. DFS returning a path

The DFS stack stores pairs `(state, previous_state)`. On removal, tuple unpacking assigns these to `current` and `previous`. The parent is recorded only when that state is actually processed, so the parent chain follows the branch selected by the stack.

A state can have several pending entries with different predecessors, but only its first processed entry is accepted. Later entries are skipped by the visited check. Each accepted predecessor was processed earlier, so the recorded parent chain leads back to the start.

The goal returns its reconstructed path. An exhausted stack returns `None`. The result depends on successor order and has no general shortest-path guarantee.


In [14]:
def dfs_path(graph, start, goal):
    stack = [(start, None)]
    visited = set()
    parent = {}
    while stack:
        current, previous = stack.pop()
        if current in visited:
            continue
        visited.add(current)
        parent[current] = previous
        if current == goal:
            return reconstruct_path(parent, current)
        for neighbour in reversed(graph.get(current, [])):
            if neighbour not in visited:
                stack.append((neighbour, current))
    return None


In [15]:
print("DFS path:", dfs_path(graph, "S", "G"))
print("No path:", dfs_path(graph, "S", "X"))
print("Already at goal:", dfs_path(graph, "S", "S"))


DFS path: ['S', 'A', 'C', 'F', 'G']
No path: None
Already at goal: ['S']


## 9. Comparison

| Question | BFS | DFS, left first |
|---|---|---|
| Frontier | Queue, FIFO | Stack, LIFO |
| Visiting order to G | S, A, B, C, G | S, A, C, F, G |
| Returned path | S → B → G | S → A → C → F → G |
| Path cost | 2 actions | 4 actions |
| Shortest here? | Yes | No |
| Uses a visited set? | Yes | Yes |

The path rows describe `bfs_path` and `dfs_path`. The visiting order is not a route: for example, A has no direct connection to B. Both algorithms terminate on these finite graphs even when no goal is reachable, because repeated states are not processed again.


In [16]:
bfs_result = bfs_path(graph, "S", "G")
dfs_result = dfs_path(graph, "S", "G")
assert bfs_result == ["S", "B", "G"]
assert dfs_result == ["S", "A", "C", "F", "G"]
assert bfs_path(graph, "S", "X") is None
assert dfs_path(graph, "S", "X") is None
print("BFS actions:", len(bfs_result) - 1)
print("DFS actions:", len(dfs_result) - 1)
print("path tests passed")


BFS actions: 2
DFS actions: 4
path tests passed


## 10. A cycle and an unreachable state

This seven-state example adds F→S and an isolated state U. There remain two routes from S to G. U exists in the graph but cannot be reached from S.

The visited check rejects S when the F→S edge is encountered. The searches therefore finish rather than endlessly traversing the cycle. U is a useful failure test: it distinguishes an existing but unreachable goal from a reachable one.


In [17]:
challenge_graph = {
    "S": ["A", "B"],
    "A": ["C"],
    "B": ["G"],
    "C": ["F"],
    "F": ["S", "G"],
    "G": [],
    "U": []
}
print("BFS reachable:", bfs(challenge_graph, "S", "G"))
print("DFS reachable:", dfs(challenge_graph, "S", "G"))
print("BFS path:", bfs_path(challenge_graph, "S", "G"))
print("DFS path:", dfs_path(challenge_graph, "S", "G"))
print("U reachable?", bfs_path(challenge_graph, "S", "U"))
assert bfs_path(challenge_graph, "S", "G") == ["S", "B", "G"]
assert dfs_path(challenge_graph, "S", "G") == ["S", "A", "C", "F", "G"]
assert bfs_path(challenge_graph, "S", "U") is None
assert dfs_path(challenge_graph, "S", "U") is None


Visiting: S
Visiting: A
Visiting: B
Visiting: C
Visiting: G
BFS reachable: True
Visiting: S
Visiting: A
Visiting: C
Visiting: F
Visiting: G
DFS reachable: True
BFS path: ['S', 'B', 'G']
DFS path: ['S', 'A', 'C', 'F', 'G']
U reachable? None


## 11. Optional extension: the Lab 05 maze

The maze uses `(row, column)` states, start `(0, 0)` and goal `(2, 3)`. `actions` lists only legal commands. `transition` rejects an illegal command instead of turning it into a self-loop. `get_successors` applies legal actions, and `get_neighbours` provides the same results.

For this small maze, a graph dictionary can be built for all free cells and passed to the existing `bfs_path` function. The path contains six coordinates, representing five legal moves at cost 1 each. This extension reuses Lab 05's problem model; it is outside the two-hour core lab.


In [18]:
maze = [
    ["S", ".", "#", "."],
    ["#", ".", "#", "."],
    [".", ".", ".", "G"],
]
START = (0, 0)
GOAL = (2, 3)
ACTIONS = {"Up": (-1, 0), "Down": (1, 0),
           "Left": (0, -1), "Right": (0, 1)}

def actions(state):
    legal = []
    row, col = state
    for action, delta in ACTIONS.items():
        dr, dc = delta
        nr, nc = row + dr, col + dc
        if 0 <= nr < len(maze) and 0 <= nc < len(maze[0]):
            if maze[nr][nc] != "#":
                legal.append(action)
    return legal

def transition(state, action):
    if action not in actions(state):
        raise ValueError("illegal action")
    dr, dc = ACTIONS[action]
    return (state[0] + dr, state[1] + dc)

def goal_test(state):
    return state == GOAL

def get_successors(state):
    successors = []
    for action in actions(state):
        successors.append(transition(state, action))
    return successors

def get_neighbours(state):
    return get_successors(state)

maze_graph = {}
for row in range(len(maze)):
    for col in range(len(maze[0])):
        if maze[row][col] != "#":
            state = (row, col)
            maze_graph[state] = get_neighbours(state)
maze_path = bfs_path(maze_graph, START, GOAL)
print(maze_path)
print("legal moves:", len(maze_path) - 1)
assert maze_path == [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3)]
assert goal_test(maze_path[-1])


[(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3)]
legal moves: 5


## Debugging and summary

BFS needs the deque import and `popleft`; DFS uses list `pop()`. The start must enter the frontier, and each call needs a fresh visited set. Indentation determines whether failure is returned after exhaustion or prematurely inside the loop.

Visited timing must match the implementation. Processing-time marking requires skipping repeated entries. The BFS path function instead marks at enqueue to prevent repeated scheduling and parent overwrites. Missing keys can be handled with `graph.get(current, [])`. Left-first DFS requires reversed push order.

Printing the current state, frontier, and visited set reveals how each step changes the search. A parent dictionary reconstructs the route, and path length is the number of edges: `len(path) - 1` for a returned path. Never calculate that length from `None` or from the visiting order.
